# Individual Statistics

Analyze individual encounter patterns over the past year:
1. Count distinct individuals grouped by species
2. Identify the most frequently encountered individuals

Set the usual environment variables (`WILDBOOK_URL`, `WILDBOOK_USERNAME`,
`WILDBOOK_PASSWORD`) before starting the kernel, or pass credentials
explicitly to `client.login()`.

In [9]:
import os
import getpass
from datetime import datetime, timedelta
from collections import Counter, defaultdict

from dotenv import load_dotenv
from pywildbook import WildbookClient
from pywildbook.queries import filter_by_date_range

load_dotenv()

if not os.environ.get("WILDBOOK_URL"):
    os.environ["WILDBOOK_URL"] = input("WILDBOOK_URL: ")
if not os.environ.get("WILDBOOK_USERNAME"):
    os.environ["WILDBOOK_USERNAME"] = input("WILDBOOK_USERNAME: ")
if not os.environ.get("WILDBOOK_PASSWORD"):
    os.environ["WILDBOOK_PASSWORD"] = getpass.getpass("WILDBOOK_PASSWORD: ")


def get_species_name(enc):
    """Extract species name from encounter, preferring taxonomy field."""
    # Try taxonomy field first (most reliable)
    taxonomy = enc.get('taxonomy', '').strip()
    if taxonomy:
        return taxonomy

    # Fall back to genus + specificEpithet
    genus = enc.get('genus', '').strip()
    species = enc.get('specificEpithet', '').strip()
    if genus or species:
        return f"{genus} {species}".strip()

    return 'Unknown'

In [10]:
client = WildbookClient()
user = client.login()
print(f"Logged in as {user['username']}")

Logged in as kirk


In [11]:
# Search for encounters from the past year
one_year_ago = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')
print(f"Searching for encounters since {one_year_ago}...")

results = client.search_encounters(
    filter_by_date_range(start_date=one_year_ago),
    size=1000,  # Adjust if you expect more encounters
    sort='date',
    sort_order='desc'
)

encounters = results.get('hits', [])
print(f"Found {len(encounters)} encounters from the past year")

Searching for encounters since 2025-02-01...
Found 1000 encounters from the past year


In [12]:
# Group individuals by species
individuals_by_species = defaultdict(set)

for enc in encounters:
    individual_id = enc.get('individualId')
    if not individual_id:
        continue

    species_name = get_species_name(enc)
    individuals_by_species[species_name].add(individual_id)

print("\n=== Distinct Individuals by Species ===")
print(f"{'Species':<30} {'Count':>10}")
print("-" * 42)

for species_name in sorted(individuals_by_species.keys()):
    count = len(individuals_by_species[species_name])
    print(f"{species_name:<30} {count:>10}")

total_individuals = sum(len(ids) for ids in individuals_by_species.values())
print("-" * 42)
print(f"{'Total':<30} {total_individuals:>10}")


=== Distinct Individuals by Species ===
Species                             Count
------------------------------------------
Equus grevyi                          257
Equus quagga                            5
------------------------------------------
Total                                 262


In [13]:
# Find most frequently encountered individuals
individual_encounters = Counter()
individual_details = {}  # Store species and display name for each individual

for enc in encounters:
    individual_id = enc.get('individualId')
    if not individual_id:
        continue

    individual_encounters[individual_id] += 1

    # Store details (first encounter wins)
    if individual_id not in individual_details:
        species_name = get_species_name(enc)
        display_name = enc.get('individualDisplayName', individual_id)
        individual_details[individual_id] = {
            'species': species_name,
            'display_name': display_name
        }

print("\n=== Most Frequently Encountered Individuals ===")
print(f"{'Display Name':<30} {'Species':<30} {'Encounters':>12}")
print("-" * 74)

for individual_id, count in individual_encounters.most_common(10):
    details = individual_details.get(individual_id, {'species': 'Unknown', 'display_name': individual_id})
    display_name = details['display_name']
    species_name = details['species']
    print(f"{display_name:<30} {species_name:<30} {count:>12}")


=== Most Frequently Encountered Individuals ===
Display Name                   Species                          Encounters
--------------------------------------------------------------------------
LWC_GZ_JAN_2025_0003           Equus grevyi                              4
LWC_GZ_MAY_2025_0012           Equus grevyi                              4
LWC_GZ_MAY_2025_0038           Equus grevyi                              4
LWC0015_24                     Equus grevyi                              4
LWC0033_24                     Equus grevyi                              3
LWC_GZ_APR_2025_0057           Equus grevyi                              3
LWC_GZ_MAY_2025_0037           Equus grevyi                              3
LWC_GZ_APR_2025_0052           Equus grevyi                              3
LWC_JAN_2025_0021              Equus grevyi                              3
LWC_GZ_MAR_2025_0087           Equus grevyi                              3


In [14]:
client.logout()

True